In [1]:
import pandas as pd
import numpy as np
from pyvolleydata.get_data import load_pbp

pd.set_option('display.max_columns', None)

In [2]:
# Gather data from pyvolleydata for all MLV seasons into one df
df = pd.DataFrame()
for yr in range(2024, 2027):
    temp = load_pbp('mlv', yr)
    df = pd.concat([df, temp])

df = df.set_index(['match_id', 'set', 'point_number'])

In [3]:
df['contact_num'] = df.groupby(level=['match_id', 'set', 'point_number']).cumcount() + 1

# isolate all receives/digs/freeball passes, block recycles, and corresponding team and contact number
passes = df['action'].isin(['R', 'D', 'F']) | ( (df['action'] == 'B') & 
         ((df['outcome'] ==  '!') | (df['outcome'] == '+') | (df['outcome'] == '-')) )
df['pass_type'] = df['action'].where(passes)
df['pass_outcome'] = df["outcome"].where(passes)
df['pass_team'] = df["team_involved"].where(passes)
df['pass_contact'] = df['contact_num'].where(passes)

# for recycled block, swap the team involved: that way the attacking team and blocking team will be the same on
# recycles, but will be different if a block touch leads to a defending dig.
recycled = (df["action"] == "B") & ((df['outcome'] ==  '!') | (df['outcome'] == '+') | (df['outcome'] == '-'))
df.loc[recycled, "pass_team"] = df.loc[recycled, "pass_team"].map({"home": "away", "away": "home"})

# fill all contacts within each rally with preceding pass and passing team
df["pass_type"] = df.groupby(["match_id", "set", "point_number"])["pass_type"].ffill()
df["pass_outcome"] = df.groupby(["match_id", "set", "point_number"])["pass_outcome"].ffill()
df["pass_team"] = df.groupby(["match_id", "set", "point_number"])["pass_team"].ffill()
df["pass_contact"] = df.groupby(["match_id", "set", "point_number"])["pass_contact"].ffill()

# debug row id to delete duplicate rows caused by rally-ending challenges
df['_row_id'] = np.arange(len(df))
"""
df['pass_source_idx'] = df['_row_id'].where(passes)
df['pass_source_idx'] = df.groupby(level=["match_id", "set", "point_number"])['pass_source_idx'].ffill()
"""

# keep only spikes spikes where spiking team and passing team are same (remove overpass kills)
spikes = (df["action"] == "A") & (df["team_involved"] == df["pass_team"])

# include passes that don't lead to a spike as the last contact in the side's sequence
play_cols = ["match_id", "set", "point_number", "play_index"]
df["play_index"] = df.groupby(play_cols[:3])["team_involved"].transform(lambda x: x.ne(x.shift()).cumsum())
df["last_action"] = df.groupby(play_cols)["action"].transform("last")
dead_passes = (df['action'].isin(['D', 'R', 'F'])) & (df['last_action'] != 'A')

# finalize data of importance
# since we have non-spike rows (dig/receive/freeball errors), make filtered dfs of spikes and non-spikes
analysis = df.loc[spikes | dead_passes].copy()

delete = [3380, 31182, 114084, 219847, 223209, 227754, 228394, 234882, 354223]
analysis = analysis[~analysis['_row_id'].isin(delete)]

attack_occurred = analysis['action'] == 'A'
no_attack = analysis['action'] != 'A'
analysis

season            match_datetime    home_team_name  \
match_id set point_number                                                       
2125267  1   1               2024  2024-01-25T01:00:00.000Z  Omaha Supernovas   
             1               2024  2024-01-25T01:00:00.000Z  Omaha Supernovas   
             2               2024  2024-01-25T01:00:00.000Z  Omaha Supernovas   
             2               2024  2024-01-25T01:00:00.000Z  Omaha Supernovas   
             2               2024  2024-01-25T01:00:00.000Z  Omaha Supernovas   
...                           ...                       ...               ...   
2464511  4   31              2026  2026-05-08T01:00:00.000Z      Dallas Pulse   
             32              2026  2026-05-08T01:00:00.000Z      Dallas Pulse   
             33              2026  2026-05-08T01:00:00.000Z      Dallas Pulse   
             34              2026  2026-05-08T01:00:00.000Z      Dallas Pulse   
             35              2026  2026-05-08T01:00:00.000Z      Dallas Pulse   

                           away_team_name team_involved  jersey_number action  \
match_id set point_number                                                       
2125267  1   1               Atlanta Vibe          away             17      A   
             1               Atlanta Vibe          home              3      D   
             2               Atlanta Vibe          home              6      A   
             2               Atlanta Vibe          away             17      A   
             2               Atlanta Vibe          away             17      A   
...                                   ...           ...            ...    ...   
2464511  4   31            San Diego Mojo          away              6      D   
             32            San Diego Mojo          away              6      A   
             33            San Diego Mojo          home              9      R   
             34            San Diego Mojo          home              9      A   
             35            San Diego Mojo          home              2      A   

                          outcome point_winner  home_score  away_score league  \
match_id set point_number                                                       
2125267  1   1                  #         away           0           1    mlv   
             1                  =         away           0           1    mlv   
             2                  -         away           0           2    mlv   
             2                  !         away           0           2    mlv   
             2                  #         away           0           2    mlv   
...                           ...          ...         ...         ...    ...   
2464511  4   31                 =         home          24           7    mlv   
             32                 #         away          24           8    mlv   
             33                 /         away          24           9    mlv   
             34                 /         away          24          10    mlv   
             35                 #         home          25          10    mlv   

                           contact_num pass_type pass_outcome pass_team  \
match_id set point_number                                                 
2125267  1   1                       4         R            +      away   
             1                       5         D            =      home   
             2                       4         R            +      home   
             2                       7         D            #      away   
             2                      10         B            !      away   
...                                ...       ...          ...       ...   
2464511  4   31                      5         D            =      away   
             32                      4         R            +      away   
             33                      2         R            /      home   
             34                      4  

In [4]:
### check that the passes we tracked are within 2 touches of the spike (logically)
check = df.loc[spikes].copy()
# check = check[check['contact_num'] - check['pass_contact'] > 2]
check['diff'] = check['contact_num'] - check['pass_contact']
check['diff'].value_counts(ascending=True)

diff
1.0     2118
2.0    89015
Name: count, dtype: int64

In [5]:
### chi squared test on distributions of pass encodings (receives, digs, freeballs)
import numpy as np
from scipy.stats import chi2_contingency

actions = ['R', 'D', 'F']
df_sub = df[df['action'].isin(actions)]

# create contingency table
contingency = pd.crosstab(df_sub['action'], df_sub['outcome'])

print(contingency)

# chi squared test
chi2, p, dof, expected = chi2_contingency(contingency)

print(f"\nChi2 = {chi2:.3f}")
print(f"p = {p:.4g}")
print(f"df = {dof}")

# cramer's V
n = contingency.to_numpy().sum()
r, k = contingency.shape

cramers_v = np.sqrt(chi2 / (n * (min(r - 1, k - 1))))

print(f"Cramer's V = {cramers_v:.3f}")

outcome      !      #      +     -     /      =
action                                         
D           12  32232    268  2330  6284  10080
F            3   6900    524   452   173    170
R        10749  11244  13378  9875  2304   2145

Chi2 = 51795.109
p = 0
df = 10
Cramer's V = 0.487


In [6]:
from statsmodels.stats.proportion import proportion_confint

LOW_N_SUPPRESS = 10
LOW_N_FLAG     = 30

def add_ci_and_flags(df, n_col, k_col): # helper function for confidence intervals and low n flags
    los, his = [], []
    for n, k in zip(df[n_col], df[k_col]):
        if n == 0:
            los.append(np.nan); his.append(np.nan)
        else:
            lo, hi = proportion_confint(k, n, method='wilson')
            los.append(lo*100); his.append(hi*100)
    df['ci_low'] = los
    df['ci_high'] = his
    df['status'] = np.select(
        [df[n_col] < LOW_N_SUPPRESS, df[n_col] < LOW_N_FLAG],
        ['suppress', 'flag'],
        default='ok'
    )
    return df

In [7]:
# denominator for non-block pass_types
non_block_mask_analysis = analysis['pass_type'] != 'B'
attack_occ_nonblock = ( analysis[non_block_mask_analysis]
    .groupby(['pass_type', 'pass_outcome'])
    .agg(n_total=('action', 'size'), n_attack=('action', lambda x: (x == 'A').sum()))
    .reset_index()
)

# denominator for B: total raw B!/B+/B- events in the original df
block_denom = ( df[(df['action'] == 'B') & (df['outcome'].isin(['!', '+', '-']))]
    .groupby('outcome')
    .size()
    .rename('n_total')
    .reset_index()
    .rename(columns={'outcome': 'pass_outcome'})
)
block_denom['pass_type'] = 'B'

# numerator for B: successfully matched recycle -> attack
block_numer = (
    analysis[analysis['pass_type'] == 'B']
    .groupby('pass_outcome')
    .agg(n_attack=('action', lambda x: (x == 'A').sum()))
    .reset_index()
)

# P(attack | pass_type, pass_outcome)
attack_occ_block = block_denom.merge(block_numer, on='pass_outcome', how='left')
attack_occ_block['n_attack'] = attack_occ_block['n_attack'].fillna(0).astype(int)
attack_occ = pd.concat([attack_occ_nonblock, attack_occ_block], ignore_index=True)
attack_occ['pct_attack'] = attack_occ['n_attack'] / attack_occ['n_total'] * 100
attack_occ = add_ci_and_flags(attack_occ, 'n_total', 'n_attack')

pd.set_option('display.width', 140)
print("=== P(attack occurs | pass grade) ===")
print(attack_occ.to_string(index=False))

=== P(attack occurs | pass grade) ===
pass_type pass_outcome  n_total  n_attack  pct_attack       ci_low   ci_high   status
        D            !       12        11   91.666667 6.461201e+01 98.513491     flag
        D            #    32232     31116   96.537602 9.633240e+01 96.731713       ok
        D            +      268       245   91.417910 8.745190e+01 94.213352       ok
        D            -     2330      1698   72.875536 7.103371e+01 74.642058       ok
        D            /     6284         0    0.000000 0.000000e+00  0.061093       ok
        D            =    10080         0    0.000000 0.000000e+00  0.038095       ok
        F            !        3         2   66.666667 2.076596e+01 93.850806 suppress
        F            #     6900      6830   98.985507 9.872030e+01 99.196198       ok
        F            +      524       513   97.900763 9.628046e+01 98.823853       ok
        F            -      452       426   94.247788 9.170518e+01 96.044627       ok
        F       

In [8]:
# P(kill | attack occurred, pass_type, pass_outcome)
# denominator: only spike rows (action == 'A') within that grade
spikes_only = analysis[analysis['action'] == 'A'].copy()
kill_rate = (
    spikes_only.groupby(['pass_type', 'pass_outcome'])
    .agg(n_attacks=('outcome', 'size'),
         n_kills=('outcome', lambda x: (x == '#').sum()),
         n_errors=('outcome', lambda x: (x == '=').sum()))
    .reset_index()
)
kill_rate['pct_kill']  = kill_rate['n_kills']  / kill_rate['n_attacks'] * 100
kill_rate['pct_error'] = kill_rate['n_errors'] / kill_rate['n_attacks'] * 100
kill_rate = add_ci_and_flags(kill_rate, 'n_attacks', 'n_kills')

attack_occ.loc[(attack_occ['pass_type'] == 'B') & (attack_occ['pass_outcome'] != '!'), 'status'] = 'suppress'

print("\n=== P(kill | attack occurred, pass grade) ===")
print(kill_rate.to_string(index=False))


=== P(kill | attack occurred, pass grade) ===
pass_type pass_outcome  n_attacks  n_kills  n_errors  pct_kill  pct_error    ci_low   ci_high   status
        B            !       5945     1692       433 28.460892   7.283431 27.328074 29.621527       ok
        B            +         22        5         1 22.727273   4.545455 10.123040 43.439953     flag
        B            -         66       20         3 30.303030   4.545455 20.554021 42.218807       ok
        D            !         11        4         2 36.363636  18.181818 15.166471 64.619883     flag
        D            #      31116     9294      2289 29.868878   7.356344 29.362853 30.379873       ok
        D            +        245       66        18 26.938776   7.346939 21.771170 32.818391       ok
        D            -       1698      382       148 22.497055   8.716137 20.574307 24.543965       ok
        F            !          2        0         0  0.000000   0.000000  0.000000 65.761977 suppress
        F            #    

In [9]:
# best to worst order of pass ratings
OUTCOME_ORDER = ['#', '+', '!', '-', '/', '=']

def make_matrix(long_df, value_col, status_col, pass_types, dp=1):
    # helper function to pivot result dfs to matrix style for my figures
    sub = long_df[long_df['pass_type'].isin(pass_types)]
    out = pd.DataFrame(index=pass_types, columns=OUTCOME_ORDER, dtype=object)
    for _, row in sub.iterrows():
        pt, po = row['pass_type'], row['pass_outcome']
        if po not in OUTCOME_ORDER:
            continue
        if row[status_col] == 'suppress':
            out.loc[pt, po] = '-'
        else:
            val = f"{row[value_col]:.{dp}f}"
            if row[status_col] == 'flag':
                val += ' *'
            out.loc[pt, po] = val
    return out.fillna('-')

# D/F/R attack occurrence matrix
attack_matrix = make_matrix(attack_occ[attack_occ['pass_type'] != 'B'], value_col='pct_attack', status_col='status', pass_types=['D', 'F', 'R'])

# D/F/R kill rate matrix
kill_matrix = make_matrix(kill_rate[kill_rate['pass_type'] != 'B'], value_col='pct_kill', status_col='status', pass_types=['D', 'F', 'R'])

# D/F/R error rate matrix
error_matrix = make_matrix(kill_rate[kill_rate['pass_type'] != 'B'], value_col='pct_error', status_col='status',pass_types=['D', 'F', 'R'])

print("=== Attack Occurrence % (R/D/F) ===")
print(attack_matrix)
print("\n=== Kill % (R/D/F) ===")
print(kill_matrix)
print("\n=== Error % (R/D/F) ===")
print(error_matrix)

# block recycle table
block_rows = []
for outcome in ['!', '-', '+']:
    a_row = attack_occ[(attack_occ['pass_type'] == 'B') & (attack_occ['pass_outcome'] == outcome)]
    k_row = kill_rate[(kill_rate['pass_type'] == 'B') & (kill_rate['pass_outcome'] == outcome)]

    a_suppressed = (not a_row.empty) and (a_row.iloc[0]['status'] == 'suppress')

    n_total   = '-' if a_suppressed else int(a_row.iloc[0]['n_total'])
    n_attacks = int(k_row.iloc[0]['n_attacks']) if not k_row.empty else 0
    pct_attack = '-' if a_suppressed else f"{a_row.iloc[0]['pct_attack']:.1f}"

    if k_row.empty:
        pct_kill = '-'
        pct_error = '-'
    else:
        pct_kill = f"{k_row.iloc[0]['pct_kill']:.1f}"
        pct_error = f"{k_row.iloc[0]['pct_error']:.1f}"
        if k_row.iloc[0]['status'] == 'flag':
            pct_kill += ' *'
            pct_error += ' *'
        elif k_row.iloc[0]['status'] == 'suppress':
            pct_kill = '-'
            pct_error = '-'

    block_rows.append({
        'block_grade': outcome,
        'n_total': n_total,
        'n_attacks': n_attacks,
        'pct_attack': pct_attack,
        'pct_kill': pct_kill,
        'pct_error': pct_error
    })

block_df = pd.DataFrame(block_rows).set_index('block_grade')
print("\n=== Block Recycle Table ===")
print(block_df)

=== Attack Occurrence % (R/D/F) ===
      #     +       !     -    /    =
D  96.5  91.4  91.7 *  72.9  0.0  0.0
F  99.0  97.9       -  94.2  0.0  0.0
R  99.4  99.3    99.3  92.4  0.0  0.0

=== Kill % (R/D/F) ===
      #     +       !     -  /  =
D  29.9  26.9  36.4 *  22.5  -  -
F  41.9  44.2       -  30.8  -  -
R  45.9  42.0    36.5  27.9  -  -

=== Error % (R/D/F) ===
     #    +       !    -  /  =
D  7.4  7.3  18.2 *  8.7  -  -
F  5.6  4.5       -  8.9  -  -
R  5.7  5.9     6.7  7.9  -  -

=== Block Recycle Table ===
            n_total  n_attacks pct_attack pct_kill pct_error
block_grade                                                 
!              7359       5945       80.8     28.5       7.3
-                 -         66          -     30.3       4.5
+                 -         22          -   22.7 *     4.5 *


In [10]:
from scipy.stats import norm

    # n: array of trial counts per ordered group
    # r: array of success counts per ordered group
    # scores: ordinal scores per group (same order as n/r), higher score = expected higher success rate direction
    # Returns Z stat and two-sided p value.
def cochran_armitage(n, r, scores):
    n = np.asarray(n, dtype=float)
    r = np.asarray(r, dtype=float)
    x = np.asarray(scores, dtype=float)

    N = n.sum()
    R = r.sum()
    p_bar = R / N

    numerator = np.sum(x * (r - n * p_bar))
    denom = np.sqrt(p_bar * (1 - p_bar) * (np.sum(n * x**2) - (np.sum(n * x)) ** 2 / N))

    Z = numerator / denom
    p = 2 * (1 - norm.cdf(abs(Z)))
    return Z, p


# ordinal scores, best-to-worst per pass_type, matching true quality order
score_maps = {
    'R': {'#': 3, '+': 2, '!': 1, '-': 0}, # full scale, well-populated
    'D': {'#': 2, '+': 1, '-': 0},         # '!' excluded: n=12, too sparse
    'F': {'#': 2, '+': 1, '-': 0},         # '!' excluded: n=2/3, too sparse
}

print("=== Cochran-Armitage trend test: kill rate by pass grade ===\n")
for pass_type, scores in score_maps.items():
    sub = kill_rate[
        (kill_rate['pass_type'] == pass_type) &
        (kill_rate['pass_outcome'].isin(scores.keys()))
    ].copy()

    # order rows to match score dict
    sub['score'] = sub['pass_outcome'].map(scores)
    sub = sub.sort_values('score')

    n = sub['n_attacks'].values
    r = sub['n_kills'].values
    x = sub['score'].values

    Z, p = cochran_armitage(n, r, x)

    print(f"--- {pass_type} ---")
    print(sub[['pass_outcome', 'n_attacks', 'n_kills', 'pct_kill']].to_string(index=False))
    print(f"Z = {Z:.3f}, p = {p:.4g}")
    direction = "positive (kill rate rises with pass quality, as expected)" if Z > 0 else \
                "negative (kill rate FALLS as pass quality rises -- check for a bug or a real surprise)"
    print(f"direction: {direction}\n")

=== Cochran-Armitage trend test: kill rate by pass grade ===

--- R ---
pass_outcome  n_attacks  n_kills  pct_kill
           -       9123     2542 27.863641
           !      10675     3896 36.496487
           +      13288     5583 42.015352
           #      11172     5131 45.927318
Z = 27.476, p = 0
direction: positive (kill rate rises with pass quality, as expected)

--- D ---
pass_outcome  n_attacks  n_kills  pct_kill
           -       1698      382 22.497055
           +        245       66 26.938776
           #      31116     9294 29.868878
Z = 6.542, p = 6.082e-11
direction: positive (kill rate rises with pass quality, as expected)

--- F ---
pass_outcome  n_attacks  n_kills  pct_kill
           -        426      131 30.751174
           +        513      227 44.249513
           #       6830     2862 41.903367
Z = 3.504, p = 0.0004588
direction: positive (kill rate rises with pass quality, as expected)



In [ ]:
### DEBUG CELL
### from when I was trying to decipher the "implicit block recycle pass" problem

df['next_contact'] = df.groupby(level=['match_id', 'set', 'point_number'])['action'].shift(-1)
df['next_team'] = df.groupby(level=['match_id', 'set', 'point_number'])['team_involved'].shift(-1)

block_recycle = df[(df['action'] == 'B') & (df['outcome'] == '!')]
block_plus = df[(df['action'] == 'B') & (df['outcome'] == '+')]
block_minus = df[(df['action'] == 'B') & (df['outcome'] == '-')]

print(f"===== BLOCK RECYCLE =====\nn = {len(block_recycle)}\n" + 
      f"% attacking team next touch = {(block_recycle['next_team'] != block_recycle['team_involved']).sum() / len(block_recycle)}\n" + 
      f"% blocking team next touch = {(block_recycle['next_team'] == block_recycle['team_involved']).sum() / len(block_recycle)}\n\n")
print(f"===== BLOCK MINUS =====\nn = {len(block_minus)}\n" + 
      f"% attacking team next touch = {(block_minus['next_team'] != block_minus['team_involved']).sum() / len(block_minus)}\n" + 
      f"% blocking team next touch = {(block_minus['next_team'] == block_minus['team_involved']).sum() / len(block_minus)}\n\n")
print(f"===== BLOCK PLUS =====\nn = {len(block_plus)}\n" + 
      f"% attacking team next touch = {(block_plus['next_team'] != block_plus['team_involved']).sum() / len(block_plus)}\n" + 
      f"% blocking team next touch = {(block_plus['next_team'] == block_plus['team_involved']).sum() / len(block_plus)}\n\n")